# Prism Config Sweep — Find the Overfitting Boundary
**Runtime → A100 or T4**

750 steps each, ~10 min per run on A100. Tests 8 configs to find
where spectral init helps without overfitting.

Each run saves to disk immediately.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/prism'):
    subprocess.run(['rm', '-rf', '/content/prism'])
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism
!pip install -q transformers datasets scipy
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Ready.')

In [ ]:
# Cell 2: Run sweep (8 configs, ~80 min on A100, ~3 hrs on T4)
import sys, os, json, gc, time, warnings
import torch
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn
from prism.spectral_init import apply_spectral_init

install_signal_handlers()
device = 'cuda'

# Pre-extract directions once (expensive, ~30s)
with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)
print('Extracting pretrained directions...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')
print('Done.')
gc.collect()

STEPS = 750
EVALS = [100, 250, 500, 750]

base = dict(
    max_steps=STEPS,
    eval_steps=EVALS,
    log_every=50,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

# Define sweep configs
configs = [
    # Baseline
    ('ortho_1x', 6.25e-5, 200, 'orthogonal', None),
    
    # Prism full (UV align) at different LRs
    ('prism_uv_0.5x', 3.125e-5, 200, 'prism_uv', 0.5),
    ('prism_uv_1x', 6.25e-5, 200, 'prism_uv', 0.5),
    ('prism_uv_1.5x', 9.375e-5, 200, 'prism_uv', 0.5),
    ('prism_uv_2x', 1.25e-4, 200, 'prism_uv', 0.5),
    
    # Spectral shape only (no UV alignment) — isolate overfitting source
    ('spectral_only_1x', 6.25e-5, 200, 'spectral', None),
    ('spectral_only_2x', 1.25e-4, 200, 'spectral', None),
    
    # Prism UV at 1x LR with longer warmup
    ('prism_uv_1x_warm500', 6.25e-5, 500, 'prism_uv', 0.5),
]

all_results = {}

for name, lr, warmup, init_type, align_strength in configs:
    result_path = f'/content/sweep_{name}.json'
    if os.path.exists(result_path):
        print(f'\n  [{name}] SKIP — already done')
        all_results[name] = json.load(open(result_path))
        continue
    
    print(f'\n{"="*60}')
    print(f'  {name} (lr={lr}, warmup={warmup}, init={init_type})')
    print(f'{"="*60}')
    
    # Build init function
    if init_type == 'orthogonal':
        init_fn = make_init_fn('orthogonal')
    elif init_type == 'spectral':
        init_fn = make_init_fn('imt_shaped', spectra_coeffs=extracted['spectra_coeffs'])
    elif init_type == 'prism_uv':
        init_fn = make_hybrid_init_fn(
            extracted['spectra_coeffs'], dirs,
            lam=1.0, align_mode='UV', align_strength=align_strength
        )
    
    config = TrainConfig(**base, lr=lr, warmup_steps=warmup)
    t0 = time.time()
    result = train(config, init_fn=init_fn, init_name=name, verbose=True)
    elapsed = time.time() - t0
    
    data = {
        'name': name, 'lr': lr, 'warmup': warmup, 'init_type': init_type,
        'final_ppl': result['final_ppl'],
        'elapsed': elapsed,
        'checkpoints': {str(k): v for k, v in result['checkpoints'].items()},
        'gpu': torch.cuda.get_device_name(0),
    }
    with open(result_path, 'w') as f:
        json.dump(data, f, indent=2)
    all_results[name] = data
    print(f'  SAVED {result_path} — val_ppl@750: {result["checkpoints"].get(750, "?")}')
    _clear_memory(device); gc.collect()

# === SUMMARY TABLE ===
print(f'\n{"="*70}')
print(f'  SWEEP RESULTS')
print(f'{"="*70}')
print(f'{"Config":>25s}  {"LR":>10s}  {"@250":>8s}  {"@500":>8s}  {"@750":>8s}  {"Overfit?":>8s}')
print('-' * 75)
for name, data in all_results.items():
    cp = data['checkpoints']
    p250 = cp.get('250', 0)
    p500 = cp.get('500', 0)
    p750 = cp.get('750', 0)
    overfit = 'YES' if p750 > p500 and p500 > 0 else 'no'
    print(f'{name:>25s}  {data["lr"]:>10.2e}  {p250:>8.1f}  {p500:>8.1f}  {p750:>8.1f}  {overfit:>8s}')

# Save combined
with open('/content/sweep_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved /content/sweep_summary.json')

In [ ]:
# Cell 3: Download
from google.colab import files
files.download('/content/sweep_summary.json')